<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/Daily_challenge_D4W6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Défi quotidien : Optimisation des LLM avec LoRA
Ce notebook implémente l'ajustement fin (fine-tuning) du modèle `bloomz-560m` en utilisant la technique LoRA (Low-Rank Adaptation) sur l'ensemble de données de citations.

In [ ]:
# Installation des bibliothèques nécessaires
!pip install -q peft==0.4.0 datasets transformers accelerate
!mkdir -p cache

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# 1. Chargement du modèle et du tokenizer
model_name = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

# 2. Chargement et prétraitement des données (échantillon de 10%)
data = load_dataset("Abirate/english_quotes", split="train[:10%]")

def tokenize_function(samples):
    return tokenizer(samples["quote"], truncation=True, padding="max_length", max_length=128)

tokenized_data = data.map(tokenize_function, batched=True)
train_sample = tokenized_data.select(range(5))

display(train_sample)

In [ ]:
import peft
from peft import LoraConfig, get_peft_model

# 3. Configuration de LoRA
# BLOOM utilise généralement 'query_key_value' comme modules cibles
lora_config = LoraConfig(
    r=1,
    lora_alpha=1,
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Appliquer les couches d'adaptation au modèle de base
peft_model = get_peft_model(foundation_model, lora_config)
peft_model.print_trainable_parameters()

In [ ]:
import transformers
from transformers import TrainingArguments, Trainer
import os
import time

# 4. Configuration de l'entraînement
output_directory = os.path.join("cache", "peft_lab_outputs")
training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate=3e-2,
    num_train_epochs=1,
    use_cpu=True, # Forcé sur CPU comme demandé dans l'indice
    logging_steps=1
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_sample,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()

In [ ]:
# 5. Sauvegarde du modèle
time_now = int(time.time())
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path)

print(f"Modèle sauvegardé dans : {peft_model_path}")

In [ ]:
# 6. Inférence avec le modèle affiné
inputs = tokenizer("Two things are infinite: ", return_tensors="pt")

# Génération
outputs = peft_model.generate(
    input_ids=inputs["input_ids"],
    max_new_tokens=20,
    no_repeat_ngram_size=2
)

print("Résultat généré :")
print(tokenizer.batch_decode(outputs, skip_special_tokens=True))